# Imports

In [80]:
import sys
import torch
import os
import numpy as np
sys.path.append('../..')
import pandas as pd
import plotly.express as px
from definitions import EXPERIMENTS_DIR
from typing import List, Dict, Tuple
from torch import Tensor
from src.commands.graph_learning.main import Trainer, Config, Graph
from src.commands.feature_extraction.graph_builder import EdgeTypes
from torch_geometric import utils as gnn_utils
from torch_scatter import scatter
from torch_geometric.data import HeteroData
from functools import cache
from src.helpers.torch_utils import TensorDict

In [2]:
px.defaults.template = "seaborn"

In [3]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn

# Utils

In [4]:
def compute_entities_scores(
    results : Dict,
    graph : Graph,
    min_importance : float = 0.05
) -> Dict[str, Tensor]:

    E2A = graph.pygeo['Entity', 'APPEARIN', 'Article'].edge_index
    r = {}

    for model, model_results in results.items():

        entities_weights = model_results['attn_weights'].flatten()

        entities_weights = gnn_utils.softmax(
            src=entities_weights[E2A[0].flatten()],
            index=E2A[1],
            dim=0,
            num_nodes=graph.pygeo['Article'].x.size(0)
        )

        entities_weights_mean = scatter(
            src=entities_weights,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='mean'
        )

        entities_weights_sum = scatter(
            src=entities_weights,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='sum'
        )

        degree = gnn_utils.degree(E2A[1], num_nodes=graph.pygeo['Article'].x.size(0))
        degree = 1 / degree
        degree = degree[E2A[1]]

        entities_weights_clipped_sum = scatter(
            src=entities_weights - min_importance,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='sum'
        )

        ### Relative Rank Score
        _, docs = E2A
        indices = torch.arange(len(docs))
        ranks = torch.zeros_like(E2A[0], dtype=torch.float)

        for doc in docs.unique():
            mask = docs == doc
            weights = entities_weights[mask]
            entities_indices = indices[mask]
            rank = weights.argsort(descending=True) + 1
            relative_rank = 1 - (rank.float() / mask.float().sum())
            ranks[entities_indices] = relative_rank

        entities_relative_rank = scatter(
            src=ranks,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='mean'
        )

        r[model] = {
            "avg" : entities_weights_mean,
            "sum" : entities_weights_sum,
            "sum_clipped" : entities_weights_clipped_sum,
            "Sigmoid" : (model_results['attn_weights'] - model_results['attn_weights'].mean()).flatten().sigmoid(),
            'rank' : entities_relative_rank
        }

    return r

In [5]:
def compute_pseudo_labels(
    graph : Graph, 
    entities_weights : Tensor, 
    min_e_score : float,
    threshold : float | str,
    alpha : float = 2, # alpha >= 1
    freq_bounds : Tuple[float, float] = (0.01, 0.2)
) -> Tensor:

    X_a = graph.pygeo['Article'].x_passage
    X_e = graph.pygeo['Entity'].x
    entities_mask = entities_weights.flatten().gt(min_e_score)

    new_indices = torch.arange(entities_mask.sum(), device=entities_weights.device)
    indices_map = torch.full_like(entities_mask, -1, dtype=torch.long)
    indices_map[entities_mask] = new_indices

    entities, articles = graph.pygeo['Entity', 'APPEARIN', 'Article'].edge_index

    articles = articles[entities_mask[entities]]
    entities = entities[entities_mask[entities]]
    entities = indices_map[entities]

    Y = X_a @ X_e[entities_mask,:].T

    if threshold == 'median':
        threshold = Y.median()
    elif threshold == 'mean':
        threshold = Y.mean()

    Y = (Y - Y.min()) / (Y.max() - Y.min() + 1e-8)
    threshold = (threshold - Y.min()) / (Y.max() - Y.min() + 1e-8)
    Y[articles, entities] = 1.0

    Y = torch.where(
        Y < threshold,
        Y ** alpha,
        1 - (1 - Y) ** alpha
    )

    freq = Y.gt(0.5).float().mean(dim=0)
    mask = (freq >= freq_bounds[0]) & (freq <= freq_bounds[1])

    Y = Y[:,mask]
    
    return Y

In [6]:
def get_label_stats(Y : Tensor, threshold : float = 0.5) -> Dict[str, float]:

    ths = [
        Y.quantile(0.25, dim=0).mean().item(), 
        Y.quantile(0.5, dim=0).mean().item(), 
        Y.quantile(0.75, dim=0).mean().item(), 
        Y.quantile(0.95, dim=0).mean().item()
    ]

    num_labels = Y.gt(threshold).sum(dim=1).float()

    quantiles = torch.quantile(
        num_labels,
        torch.tensor([0, .25, .5, .75, .95], device=Y.device)
    )

    return {
        "sparsity" : Y.gt(threshold).float().mean().item(),
        "Average Num. Docs / Label" : Y.gt(threshold).sum(dim=0).float().mean().item(),
        "Average Num. Labels / Doc" : Y.gt(threshold).sum(dim=1).float().mean().item(),
        "Mean" : Y.mean().item(),
        "Avg. Labels / Doc at Label-Value Quantiles [0,25; 0.5, 0.75, 0.95]" : [Y.gt(th).sum(dim=1).float().mean().item() for th in ths],
        "Quantiles [0,25; 0.5, 0.75, 0.95]" : ths,
        "Distribution of labels per document" : quantiles,
        "Num. Doc with zero labels" : Y.gt(threshold).sum(dim=-1).eq(0).sum().item(),
        "Num. Labels with zero doc (0)" : Y.gt(threshold).sum(dim=0).eq(0).sum().item(),
        "D" : torch.where(Y < 0.5, Y, 1 - Y).mean().item()
    }

In [7]:
def load_histories(path : str) -> pd.DataFrame:

    histories_dir = os.path.join(path, 'histories')
    histories = []

    for filename in os.listdir(histories_dir):
        filename = os.path.join(histories_dir, filename)
        df = pd.read_csv(filename)
        df['iteration'] = np.arange(len(df))
        histories.append(df)

    histories = pd.concat(histories)

    return histories #.groupby(by=['epoch', 'phase', 'config']).mean().reset_index()

In [8]:
def load_results(path : str) -> Dict[str, Dict[str, Tensor]]:

    results_dir = os.path.join(path, 'results')
    results = {}

    for filename in os.listdir(results_dir):
        result_path = os.path.join(results_dir, filename)
        exp = filename.split('.')[0].replace('predictions_', '')
        results[exp] = torch.load(result_path, map_location='cpu', weights_only=False)

    return results

In [9]:
@cache
def load_data(experiment : str) -> Tuple[Graph, HeteroData]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = trainer.load_data(config=config.training_config, return_raw=True)

    return graph, data

In [10]:
@cache
def load_model(experiment : str, config_name : str) -> Tuple[Graph, HeteroData, torch.nn.Module]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = load_data(experiment=experiment)

    model_config = config.training_config.configs[config_name]
    model = trainer.create_model(config=model_config, data=graph)

    return model

In [11]:
def get_dataframes(graph : Graph) -> List[pd.DataFrame]:

    articles_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id,  **x.properties }, graph.get_nodes('Article').nodes_generator))).set_index('id')

    entities_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id, **x.properties }, graph.get_nodes('Entity').nodes_generator)))
    
    triples_df = pd.DataFrame(data=list(map(lambda x : {
        "id" : x.id,
        "src" : x.src,
        "dst" : x.dst,
        "predicate" : x.properties['predicate']
    }, graph.get_edges(EdgeTypes.Entity_HASRELATIONWITH_Entity).edges_generator)))

    articles_df = articles_df \
        .join(pd.DataFrame(graph._edges[EdgeTypes.Article_HASKEYWORD_Keyword].edges_generator)[['src', 'dst']] \
        .groupby('src').agg(list)) \
        .rename(columns={'dst':'keywords'})

    return articles_df, entities_df, triples_df

In [12]:
def style_paper_plot(
    fig,
    *,
    width=950,
    height=600,
    font_family="Arial",
    font_size=14,
):
    fig.update_layout(
        width=width,
        height=height,
        font=dict(
            family=font_family,
            size=font_size,
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(
            l=80,
            r=30,
            t=30,
            b=70,
        ),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="lightgray",
        gridwidth=0.5,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    return fig

In [13]:
EXPERIMENT = "topic_modeling/entities_learning"
EXPERIMENT_PATH = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
config = Config(experiment=EXPERIMENT)

In [14]:
history = load_histories(EXPERIMENT_PATH)
results = load_results(EXPERIMENT_PATH)

In [15]:
graph, data = load_data(experiment=EXPERIMENT)
# model = load_model(experiment=EXPERIMENT, config_name="baseline")

/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/torch_sparse/matmul.py:97: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  C = torch.sparse.mm(A, B)


In [16]:
articles_df, entities_df, triples_df = get_dataframes(graph)

In [17]:
entities_weights = compute_entities_scores(
    results=results,
    graph=graph,
    min_importance=0.05
)

In [18]:
# entities_weights['Attn']['Sigmoid_gated_clipped_sum'] = entities_weights['Attn']['Sigmoid'] * entities_weights['Attn']['sum_clipped']

for k, v in entities_weights['baseline_2_final'].items():
    entities_df[f'{k}_Score'] = v.numpy()
    entities_df[f'{k}_Rank'] = (-entities_df[f'{k}_Score']).rank().to_numpy()

In [19]:
subset_df = entities_df[entities_df['id'].isin([
    'knowledge_graph', 'deep_learning', 'graph_neural_network', 'internet_of_thing', 
    'model', 'data', 'france', 'nante', 'method', 'dataset', 'existing_cable_driven_parallel_robot',
    'quality', 'image_processing', 'video_quality', 'video_encoding', 'deepfake_detection_task',
    'state_of_the_art', 'state_of_the_art_method', 'sparql_query', 'food_waste', 'methodology',
])]

subset_df.drop(columns=['id', 'relevancy_scores', 'generality_scores', 'out_degree', 'in_degree', 'degree', 'pagerank', 'specificity_weighted_pagerank'])

,name,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
46,nante,0.039267,46653.0,0.274868,5887.0,-0.075132,57020.0,0.371141,48523.0,0.608102,19206.0
47,france,0.062920,32494.0,3.838132,2.0,0.788132,104.0,0.359410,49878.0,0.398524,34587.0
91,method,0.026941,53933.0,1.239298,78.0,-1.060702,58183.0,0.136068,58192.0,0.327342,39698.0
275,graph neural network,0.086901,22396.0,1.216615,83.0,0.516615,587.0,0.483201,30517.0,0.390975,34854.0
337,internet of thing,0.044469,43243.0,2.312378,11.0,-0.287622,58041.0,0.337669,52211.0,0.433309,31621.0
944,model,0.022128,55940.0,2.566904,8.0,-3.233098,58198.0,0.157904,58165.0,0.377471,35670.0
2530,deep learning,0.039131,46738.0,1.917407,23.0,-0.532593,58152.0,0.296577,55273.0,0.466732,28973.0
2789,quality,0.014443,57701.0,0.202207,10086.0,-0.497793,58143.0,0.148973,58180.0,0.439647,31127.0
3347,dataset,0.023942,55274.0,0.766132,455.0,-0.833868,58175.0,0.140938,58189.0,0.408503,33529.0
3423,state of the art,0.025239,54705.0,0.555250,1231.0,-0.544750,58156.0,0.204645,57934.0,0.364925,36601.0


In [20]:
entities_df[entities_df.columns[entities_df.columns.str.endswith('_Rank')]].corr()

,avg_Rank,sum_Rank,sum_clipped_Rank,Sigmoid_Rank,rank_Rank
avg_Rank,1.000000,0.783074,0.971843,0.645464,0.035875
sum_Rank,0.783074,1.000000,0.791223,0.366395,0.009256
sum_clipped_Rank,0.971843,0.791223,1.000000,0.606207,0.031314
Sigmoid_Rank,0.645464,0.366395,0.606207,1.000000,0.108402
rank_Rank,0.035875,0.009256,0.031314,0.108402,1.000000


In [21]:
entities_df[entities_df.columns[entities_df.columns.str.endswith('_Score')]].corr()

,avg_Score,sum_Score,sum_clipped_Score,Sigmoid_Score,rank_Score
avg_Score,1.000000,0.605398,0.832561,0.519568,0.037508
sum_Score,0.605398,1.000000,0.639136,0.184538,0.004569
sum_clipped_Score,0.832561,0.639136,1.000000,0.460637,0.031139
Sigmoid_Score,0.519568,0.184538,0.460637,1.000000,0.107252
rank_Score,0.037508,0.004569,0.031139,0.107252,1.000000


In [22]:
def print_articles_entities_importance(
    article_id : int,
    articles_df : pd.DataFrame,
    entities_df : pd.DataFrame,
    score_col : str,
    graph : Graph
):

    article = articles_df.iloc[article_id]
    title = article['title']

    entities_ids, articles_ids = graph.pygeo['Entity', 'APPEARIN', 'Article'].edge_index
    entities_ids = entities_ids[articles_ids == article_id].tolist()

    entities = entities_df.iloc[entities_ids]
    entities = pd.DataFrame.sort_values(entities, by=score_col, ascending=False)

    print(f"Title : {title} ")

    for idx, row in entities.iterrows():
        print(f"{row['name']} {{ 'score' : {row[score_col]} }}")

In [23]:
def print_nn_entities(
    entity_id : int,
    embeddings : Tensor,
    entities_df : pd.DataFrame,
    k : int = 5
):

    embeddings = embeddings / embeddings.norm(p=2, dim=-1, keepdim=True)
    s = embeddings[[entity_id],:] @ embeddings.T
    i = s.argsort(descending=True).flatten().tolist()

    print(dict(zip(
        entities_df.iloc[i].head(k)['name'].tolist(),
        s[:,i].flatten().tolist()
    )))

In [24]:
print_nn_entities(
    entity_id=3,
    embeddings=results['baseline_2_final']['entities_sem_embeddings'],
    entities_df=entities_df,
    k=5
)

{'global warming potential': 1.0, 'global warming': 0.6778606176376343, 'climate change': 0.5613200068473816, 'major cause of global warming': 0.5472866296768188, 'potential impact of climate change': 0.5397343635559082}


In [25]:
entities_df[entities_df['name'].str.contains('emission')]

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
1,global_greenhouse_gas_emission,global greenhouse gas emission,0.454730,0.379704,0.0,2.0,2.0,0.006289,0.038371,0.094402,20057.0,0.188805,11272.0,0.088805,13318.0,0.519792,24575.0,0.423529,32433.5
11,sea_surface_intrinsic_infrared_emission,sea surface intrinsic infrared emission,0.141335,0.000002,0.0,1.0,1.0,0.003384,0.053183,0.297184,2154.0,0.297184,5051.0,0.247184,3216.0,0.670163,6222.0,0.000000,56570.5
296,embodied_greenhouse_gasses_emission_model,embodied greenhouse gasses emission model,0.440828,0.000002,1.0,0.0,1.0,0.002016,0.042750,0.163674,8061.0,0.163674,14015.0,0.113674,10235.0,0.716083,3232.0,0.642857,17101.0
519,fluorescence_emission,fluorescence emission,0.433493,0.444309,2.0,1.0,3.0,0.016170,0.062933,0.086800,22425.0,0.260401,6498.0,0.110401,10581.0,0.416593,41850.0,0.329545,39654.0
1117,greenhouse_gas_emission,greenhouse gas emission,0.525192,0.800881,11.0,21.0,32.0,0.175416,0.250705,0.051945,38621.0,1.142782,106.0,0.042782,22894.0,0.429667,39693.0,0.403271,33738.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57070,emission_inventory,emission inventory,0.604700,0.000002,4.0,1.0,5.0,0.021218,0.057966,0.074908,26930.0,0.074908,34318.0,0.024908,28880.0,0.502412,27402.0,0.916667,2030.0
57202,redshifted_emission,redshifted emission,0.506057,0.000002,1.0,2.0,3.0,0.012403,0.048568,0.063424,32238.0,0.063424,38894.0,0.013424,33483.0,0.549415,20032.0,0.235294,45068.0
57455,emission_scenario,emission scenario,0.647638,0.000001,0.0,1.0,1.0,0.001277,0.037119,0.027089,53853.0,0.027089,55008.0,-0.022911,51003.0,0.414038,42272.0,0.120000,51171.5
57554,steady_state_emission_spectroscopy,steady state emission spectroscopy,0.436173,0.000001,1.0,0.0,1.0,0.001082,0.035626,0.311589,1865.0,0.311589,4565.0,0.261589,2844.0,0.476030,31707.0,0.250000,44198.5


In [26]:
fig = px.histogram(
    data_frame=entities_df,
    x='avg_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Average Document Attention Score")

fig.show()

In [27]:
fig = px.histogram(
    data_frame=entities_df,
    x='sum_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Sum Document Attention Score")

fig.show()

In [28]:
fig = px.histogram(
    data_frame=entities_df,
    x='sum_clipped_Score',
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Sum clipped Attention Score")

fig.show()

In [29]:
fig = px.histogram(
    data_frame=entities_df,
    x='Sigmoid_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Overall Attention Score")

fig.show()

In [30]:
fig = px.histogram(
    data_frame=entities_df,
    x='rank_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Average Relative Rank")

fig.show()

In [31]:
def line_plot(entities_df : pd.DataFrame, col : str, n_steps : int = 20):

    data = []

    thresholds = np.arange(
        entities_df[col].min().item(), 
        entities_df[col].max().item(), 
        (entities_df[col].max().item() - entities_df[col].min().item()) / n_steps
    )

    for th in thresholds:

        N = (entities_df[col] >= th).astype(float).sum()
        p = 100 * (N / entities_df.shape[0])

        data.append({
            "threshold" : th,
            "%Entity" : p,
            'text' : f"{p:.2f}% ({N})"
        })

    data = pd.DataFrame(data)

    fig = px.line(
        data_frame=data,
        x="threshold",
        y="%Entity",
        markers=True,
    )

    fig = style_paper_plot(fig)
    fig.update_layout(xaxis_title=f"Threshold ({col.replace('_',' ')})", yaxis_title="%Entity (Size)")

    return fig

In [32]:
fig = line_plot(entities_df=entities_df, col='avg_Score', n_steps=50)
fig.show()

In [33]:
fig = line_plot(entities_df=entities_df, col='sum_Score', n_steps=50)
fig.show()

In [34]:
fig = line_plot(entities_df=entities_df, col='sum_clipped_Score', n_steps=200)
fig.show()

In [35]:
data['Entity'].x.size()

torch.Size([58198, 768])

In [36]:
mask  = entities_df['sum_clipped_Score'].gt(0).to_numpy()
subset = entities_df.iloc[mask]
subset[subset['name'].str.contains('sparql')].head()

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
3792,sparql_federation_engine,sparql federation engine,0.230511,0.237268,2.0,2.0,4.0,0.026964,0.086214,0.361398,1266.0,0.722796,578.0,0.622796,315.0,0.564268,17904.0,0.083333,52780.0
3793,sparql_endpoint,sparql endpoint,0.345308,0.477322,6.0,2.0,8.0,0.054703,0.123241,0.073734,27393.0,0.442402,2125.0,0.142402,7723.0,0.445502,36971.0,0.571429,21575.0
3808,online_public_sparql_endpoint,online public sparql endpoint,0.352153,0.522126,2.0,5.0,7.0,0.043115,0.102172,0.114156,15235.0,0.684938,681.0,0.384938,1272.0,0.471138,32550.0,0.479167,28249.0
3812,sparql_query_engine,sparql query engine,0.363612,0.337360,0.0,2.0,2.0,0.010079,0.056607,0.213382,4718.0,0.426764,2332.0,0.326764,1847.0,0.661352,6914.0,0.471429,28697.5
9176,sparqllm,sparqllm,0.267336,0.000002,7.0,0.0,7.0,0.040039,0.094353,0.059914,34042.0,0.059914,40394.0,0.009914,35015.0,0.417667,41681.0,0.266667,43215.5


In [37]:
fig = line_plot(entities_df=entities_df, col='Sigmoid_Score', n_steps=50)
fig.show()

In [38]:
fig = line_plot(entities_df=entities_df, col='rank_Score', n_steps=50)
fig.show()

### Visualiza Top-Entities by each metric

In [39]:
entities_df.sort_values(by='avg_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
55481,modeling_the_remediation_of_organic_contaminant,modeling the remediation of organic contaminant,0.344924,1.700144e-06,0.0,1.0,1.0,0.001510,0.038895,0.936762,1.0,0.936762,220.0,0.886762,52.0,0.712075,3434.0,0.5,26626.5
35984,cesium_inserted_chromium_telluride,cesium inserted chromium telluride,0.240298,1.700144e-06,0.0,1.0,1.0,0.000951,0.034628,0.904671,2.0,0.904671,251.0,0.854671,58.0,0.738610,2249.0,0.5,26626.5
39470,wide_range_of_volatile_compound,wide range of volatile compound,0.460983,5.667147e-07,0.0,1.0,1.0,0.002578,0.047038,0.898908,3.0,0.898908,255.0,0.848908,60.0,0.696916,4346.0,0.5,26626.5
56840,single_redox_dmpc_liposome_collision,single redox dmpc liposome collision,0.130882,1.700144e-06,0.0,1.0,1.0,0.003928,0.057331,0.887249,4.0,0.887249,266.0,0.837249,66.0,0.826964,273.0,0.5,26626.5
57652,traditional_microstrip_printing_technology,traditional microstrip printing technology,0.318135,1.700144e-06,1.0,0.0,1.0,0.001650,0.039963,0.884944,5.0,0.884944,269.0,0.834944,68.0,0.745166,1978.0,0.5,26626.5
31558,pre_trained_language_model_camembert,pre trained language model camembert,0.382875,1.700144e-06,0.0,1.0,1.0,0.001627,0.039783,0.869472,6.0,0.869472,280.0,0.819472,74.0,0.669174,6300.0,0.5,26626.5
34343,app_development_life_cycle,app development life cycle,0.489060,1.133429e-06,0.0,1.0,1.0,0.000616,0.032073,0.869362,7.0,0.869362,281.0,0.819362,75.0,0.722358,2943.0,0.5,26626.5
43073,geometrically_exact_sliding_cosserat_rod,geometrically exact sliding cosserat rod,0.187560,1.133429e-06,0.0,1.0,1.0,0.003065,0.050751,0.867975,8.0,0.867975,284.0,0.817975,77.0,0.668179,6354.0,0.5,26626.5
22840,right_hand_side_of_the_equation,right hand side of the equation,0.318836,1.133429e-06,0.0,1.0,1.0,0.002858,0.049173,0.865012,9.0,0.865012,288.0,0.815012,84.0,0.535118,22244.0,0.5,26626.5
22088,visual_walkability,visual walkability,0.126209,1.983502e-06,0.0,1.0,1.0,0.001456,0.038480,0.855054,10.0,0.855054,295.0,0.805054,89.0,0.657206,7264.0,0.5,26626.5


In [40]:
entities_df.sort_values(by='sum_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
5566,existing_cable_driven_parallel_robot,existing cable driven parallel robot,0.294597,0.499862,87.0,20.0,107.0,0.558266,0.613138,0.105438,17178.0,5.166472,1.0,2.716471,1.0,0.525916,23630.0,0.464872,29236.0
47,france,france,0.727668,0.978004,12.0,53.0,65.0,0.400098,0.488582,0.062920,32494.0,3.838132,2.0,0.788132,104.0,0.359410,49878.0,0.398524,34587.0
385,consumption_of_energy,consumption of energy,0.645794,0.895467,49.0,100.0,149.0,0.805453,0.832069,0.033721,50160.0,3.540653,3.0,-1.709346,58192.0,0.305734,54733.0,0.436971,31401.0
504,machine_learning,machine learning,0.517023,0.839875,76.0,43.0,119.0,0.651721,0.702278,0.042668,44408.0,2.944108,4.0,-0.505892,58145.0,0.295811,55326.0,0.417566,32743.0
5881,reverberation_chamber,reverberation chamber,0.379326,0.592198,16.0,15.0,31.0,0.207623,0.302272,0.133272,11742.0,2.665438,5.0,1.665438,4.0,0.500300,27720.0,0.454531,30160.0
1545,virtual_reality,virtual reality,0.494649,0.749809,63.0,26.0,89.0,0.535280,0.611986,0.058552,34783.0,2.634851,6.0,0.384851,1273.0,0.340750,51938.0,0.456324,29806.0
1166,neural_network,neural network,0.509992,0.821132,31.0,25.0,56.0,0.345749,0.436145,0.055352,36639.0,2.601553,7.0,0.251552,3089.0,0.293607,55452.0,0.414976,33112.0
944,model,model,0.739256,0.989416,89.0,90.0,179.0,0.999989,0.999995,0.022128,55940.0,2.566904,8.0,-3.233098,58198.0,0.157904,58165.0,0.377471,35670.0
3382,unit_cell,unit cell,0.590622,0.651918,36.0,16.0,52.0,0.333720,0.428578,0.079447,25046.0,2.383424,9.0,0.883424,53.0,0.343929,51654.0,0.396794,34625.0
6668,specific_optical_property,specific optical property,0.535549,0.764530,8.0,43.0,51.0,0.318804,0.410810,0.064841,31512.0,2.334262,10.0,0.534262,528.0,0.340883,51926.0,0.385291,35133.0


In [41]:
entities_df.sort_values(by='sum_clipped_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
5566,existing_cable_driven_parallel_robot,existing cable driven parallel robot,0.294597,4.998617e-01,87.0,20.0,107.0,0.558266,0.613138,0.105438,17178.0,5.166472,1.0,2.716471,1.0,0.525916,23630.0,0.464872,29236.0
24660,fixed_time_converging_estimation_algorithm,fixed time converging estimation algorithm,0.129024,5.227660e-03,2.0,0.0,2.0,0.006817,0.040910,0.652363,167.0,1.957089,20.0,1.807089,2.0,0.643302,8564.0,0.666667,15619.0
7198,advanced_reflectarray_antenna_system,advanced reflectarray antenna system,0.254558,8.690146e-02,1.0,1.0,2.0,0.010194,0.057157,0.642669,180.0,1.928008,22.0,1.778008,3.0,0.628004,10077.0,0.500000,26626.5
5881,reverberation_chamber,reverberation chamber,0.379326,5.921977e-01,16.0,15.0,31.0,0.207623,0.302272,0.133272,11742.0,2.665438,5.0,1.665438,4.0,0.500300,27720.0,0.454531,30160.0
38240,rigourous_calibration_procedure,rigourous calibration procedure,0.429082,2.228238e-02,1.0,0.0,1.0,0.002259,0.044606,0.844314,16.0,1.688628,39.0,1.588628,5.0,0.672046,6055.0,0.500000,26626.5
34597,dex_escape,dex escape,0.500018,9.772599e-02,1.0,0.0,1.0,0.003033,0.050506,0.756686,70.0,1.513372,50.0,1.413372,6.0,0.486126,30069.0,0.500000,26626.5
12302,run_length_property,run length property,0.531287,4.601103e-01,6.0,11.0,17.0,0.099243,0.169184,0.197090,5552.0,1.773812,29.0,1.323812,7.0,0.512382,25801.0,0.563993,22083.0
5738,word_error_rate,word error rate,0.457107,3.810621e-01,5.0,3.0,8.0,0.044916,0.099697,0.259253,3008.0,1.555520,46.0,1.255520,8.0,0.422556,40869.0,0.350000,37789.5
24610,joint_video_experts_team,joint video experts team,0.497371,2.363609e-01,3.0,0.0,3.0,0.006888,0.027542,0.298931,2113.0,1.494654,51.0,1.244654,9.0,0.778501,990.0,0.772979,9437.0
20730,nudge,nudge,0.588057,3.476347e-01,5.0,4.0,9.0,0.059469,0.127408,0.462863,595.0,1.388589,61.0,1.238589,10.0,0.515051,25355.0,0.333333,38285.5


In [42]:
entities_df.sort_values(by='Sigmoid_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
47289,faculty_student_exchange,faculty student exchange,0.756229,2.266859e-06,0.0,1.0,1.0,0.003706,0.055638,0.307457,1945.0,0.307457,4694.0,0.257457,2944.0,0.940674,1.0,0.358974,37093.0
17420,new_thin_film_material,new thin film material,0.492234,1.983502e-06,0.0,1.0,1.0,0.002833,0.048979,0.216397,4559.0,0.216397,9028.0,0.166397,6214.0,0.934170,2.0,0.015873,54938.0
40189,division_by_ten,division by ten,0.800649,1.133429e-06,0.0,1.0,1.0,0.002589,0.047121,0.267273,2802.0,0.534546,1355.0,0.434546,957.0,0.933348,3.0,0.592593,20299.5
9097,willingness_to_eat_these_food,willingness to eat these food,0.518743,1.133429e-06,0.0,1.0,1.0,0.000036,0.027651,0.214206,4684.0,0.214206,9203.0,0.164206,6340.0,0.927120,4.0,0.800000,7975.0
29410,cities_of_the_future,cities of the future,0.542381,1.700144e-06,0.0,1.0,1.0,0.001727,0.040546,0.266208,2827.0,0.266208,6249.0,0.216208,4102.0,0.925400,5.0,0.625000,18176.0
33454,editors_from_mdpi,editors from mdpi,0.829420,1.133429e-06,0.0,1.0,1.0,0.001433,0.038306,0.195884,5617.0,0.195884,10597.0,0.145884,7464.0,0.925163,6.0,0.492308,27778.0
36166,insufficient_number_of_engineer,insufficient number of engineer,0.665707,1.700144e-06,4.0,1.0,5.0,0.017748,0.047729,0.165505,7900.0,0.165505,13792.0,0.115505,10047.0,0.921353,7.0,0.774648,9416.5
23391,immutable_and_transparent_log,immutable and transparent log,0.642295,1.700144e-06,0.0,1.0,1.0,0.001116,0.035886,0.223180,4266.0,0.223180,8581.0,0.173180,5857.0,0.915549,8.0,0.431818,31663.0
13001,chontaduro_seed,chontaduro seed,0.350465,1.700144e-06,0.0,1.0,1.0,0.001247,0.036886,0.299260,2101.0,0.299260,4968.0,0.249260,3148.0,0.914368,9.0,0.520000,25085.5
40128,regional_monographs_of_megalopolis,regional monographs of megalopolis,0.717908,1.133429e-06,0.0,1.0,1.0,0.003033,0.050509,0.182081,6542.0,0.182081,11933.0,0.132081,8539.0,0.913167,10.0,0.227273,45500.5


In [43]:
entities_df.sort_values(by='rank_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
36672,reconfigurable_dual_arm_configuration,reconfigurable dual arm configuration,0.402588,0.000002,4.0,0.0,4.0,0.035458,0.114111,0.021962,55985.0,0.021962,56501.0,-0.028038,52837.0,0.615719,11442.0,0.986486,1.0
51325,main_powder_physical_property,main powder physical property,0.462220,0.000002,2.0,4.0,6.0,0.036904,0.094016,0.014130,57746.0,0.014130,57809.0,-0.035870,54701.0,0.485598,30153.0,0.985915,2.0
33440,catenary,catenary,0.719761,0.000001,1.0,0.0,1.0,0.004025,0.058071,0.021093,56227.0,0.021093,56687.0,-0.028907,53077.0,0.571038,16988.0,0.984615,3.0
8149,labsar_algorithm,labsar algorithm,0.619178,0.000001,0.0,1.0,1.0,0.003041,0.050566,0.013156,57854.0,0.013156,57894.0,-0.036844,54848.0,0.454454,35435.0,0.982143,4.0
57900,battery_engineering,battery engineering,0.425415,0.000002,0.0,3.0,3.0,0.010968,0.043097,0.015970,57470.0,0.015970,57600.0,-0.034030,54362.0,0.417249,41743.0,0.980000,5.0
20890,heritage_policy,heritage policy,0.749323,0.013635,1.0,0.0,1.0,0.002856,0.049156,0.031801,51331.0,0.063602,38813.0,-0.036398,54784.0,0.660148,7027.0,0.979515,6.0
7514,new_profession,new profession,0.696993,0.000002,1.0,0.0,1.0,0.000287,0.029564,0.050989,39223.0,0.050989,44484.0,0.000989,39337.0,0.690986,4727.0,0.978723,7.0
19786,electronic_carrier_density,electronic carrier density,0.384615,0.000002,0.0,1.0,1.0,0.003073,0.050810,0.017087,57237.0,0.017087,57413.0,-0.032913,54113.0,0.451812,35904.0,0.978261,8.0
40122,field_shot,field shot,0.728159,0.000001,5.0,0.0,5.0,0.025390,0.070273,0.012724,57894.0,0.012724,57924.0,-0.037276,54907.0,0.423602,40698.0,0.977273,9.5
23384,supply_chain_digitisation,supply chain digitisation,0.254728,0.000002,2.0,0.0,2.0,0.006806,0.040860,0.024599,54978.0,0.024599,55793.0,-0.025401,51941.0,0.544400,20807.0,0.977273,9.5


# Visualize bottom entities by each metric

In [44]:
entities_df.sort_values(by='Sigmoid_Score', ascending=True).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
31495,simulate,simulate,0.597901,4.171480e-01,2.0,0.0,2.0,0.009046,0.051635,0.016074,57443.0,0.048222,45762.0,-0.101778,57408.0,0.095428,58198.0,0.138889,50287.0
36016,specific_methodology,specific methodology,0.581624,5.667147e-07,1.0,0.0,1.0,0.002737,0.048251,0.006619,58183.0,0.006619,58183.0,-0.043381,55554.0,0.109171,58197.0,0.920000,1859.0
32218,test,test,0.782064,8.589551e-01,3.0,4.0,7.0,0.043442,0.103004,0.009223,58113.0,0.046117,46704.0,-0.203883,57923.0,0.117788,58196.0,0.390606,34869.0
12855,calculation,calculation,0.761406,8.668795e-01,5.0,5.0,10.0,0.057929,0.117961,0.014014,57756.0,0.070069,36138.0,-0.179931,57875.0,0.118752,58195.0,0.426627,32321.0
54616,difference,difference,0.781523,1.700144e-06,1.0,0.0,1.0,0.002311,0.044997,0.014885,57621.0,0.014885,57712.0,-0.035115,54549.0,0.125871,58194.0,0.727273,12035.5
56077,overall_parameter,overall parameter,0.531149,1.700144e-06,2.0,0.0,2.0,0.002681,0.021010,0.019265,56759.0,0.019265,57069.0,-0.030735,53594.0,0.132316,58193.0,0.100000,52104.5
91,method,method,0.692720,9.951968e-01,46.0,21.0,67.0,0.379344,0.458951,0.026941,53933.0,1.239298,78.0,-1.060702,58183.0,0.136068,58192.0,0.327342,39698.0
29335,controlled,controlled,0.785853,1.700144e-06,1.0,1.0,2.0,0.009706,0.054809,0.008553,58138.0,0.008553,58141.0,-0.041447,55402.0,0.138254,58191.0,0.500000,26626.5
2898,measurement,measurement,0.711675,8.740606e-01,25.0,32.0,57.0,0.271066,0.336951,0.020233,56489.0,0.890258,262.0,-1.309742,58188.0,0.138742,58190.0,0.360894,37019.0
3347,dataset,dataset,0.642866,8.896889e-01,24.0,23.0,47.0,0.293682,0.385516,0.023942,55274.0,0.766132,455.0,-0.833868,58175.0,0.140938,58189.0,0.408503,33529.0


In [45]:
entities_df.sort_values(by='rank_Score', ascending=True).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
18700,customer_knowledge,customer knowledge,0.497293,0.000001,0.0,1.0,1.0,0.003718,0.055727,0.051412,38932.0,0.051412,44276.0,0.001412,39107.0,0.392004,45719.0,0.0,56570.5
42633,quasi_epitaxial_condition,quasi epitaxial condition,0.411222,0.000002,0.0,1.0,1.0,0.002089,0.043308,0.112993,15498.0,0.112993,22883.0,0.062993,18003.0,0.583702,15284.0,0.0,56570.5
50509,highest_vacancy_probability,highest vacancy probability,0.537176,0.000002,0.0,1.0,1.0,0.003829,0.056574,0.133108,11763.0,0.133108,18619.0,0.083108,14164.0,0.650271,7888.0,0.0,56570.5
45905,limited_dynamic_medium,limited dynamic medium,0.495782,0.000002,0.0,1.0,1.0,0.002111,0.043473,0.273400,2654.0,0.273400,5944.0,0.223400,3875.0,0.545261,20676.0,0.0,56570.5
35665,computation_time_reduction,computation time reduction,0.610089,0.000002,0.0,3.0,3.0,0.021016,0.081410,0.033903,50057.0,0.033903,52380.0,-0.016097,47985.0,0.264446,56792.0,0.0,56570.5
21733,seg_grad_cam_analysis,seg grad cam analysis,0.457692,0.000002,1.0,0.0,1.0,0.002684,0.047844,0.145559,10056.0,0.145559,16520.0,0.095559,12365.0,0.618145,11177.0,0.0,56570.5
43324,guide,guide,0.774199,0.000002,1.0,0.0,1.0,0.004570,0.062225,0.025881,54423.0,0.025881,55403.0,-0.024119,51467.0,0.319821,53730.0,0.0,56570.5
43329,compact_phase_space,compact phase space,0.453280,0.000001,1.0,1.0,2.0,0.008097,0.047069,0.235300,3791.0,0.235300,7792.0,0.185300,5300.0,0.489820,29455.0,0.0,56570.5
52993,web_server,web server,0.576189,0.000002,1.0,0.0,1.0,0.001448,0.038420,0.085319,22919.0,0.085319,30574.0,0.035319,25191.0,0.481451,30818.0,0.0,56570.5
49873,configuration_of_bridge,configuration of bridge,0.582384,0.000001,2.0,0.0,2.0,0.007067,0.042112,0.059074,34490.0,0.059074,40769.0,0.009074,35412.0,0.516681,25084.0,0.0,56570.5


In [46]:
entities_df.sort_values(by='avg_Score', ascending=True).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
18563,chemical_measurement,chemical measurement,0.669614,1.133429e-06,0.0,1.0,1.0,0.003756,0.056023,0.004536,58198.0,0.004536,58198.0,-0.045464,55707.0,0.275569,56342.0,0.602564,19344.0
36169,electronics_application,electronics application,0.627972,1.700144e-06,0.0,3.0,3.0,0.007808,0.031050,0.004630,58197.0,0.004630,58197.0,-0.045370,55697.0,0.246842,57264.0,0.000000,56570.5
51328,surface_mechanical_property,surface mechanical property,0.513473,1.700144e-06,1.0,0.0,1.0,0.002279,0.044757,0.004859,58196.0,0.004859,58196.0,-0.045141,55682.0,0.245072,57313.0,0.957747,244.0
36659,control_operation,control operation,0.777993,1.700144e-06,0.0,1.0,1.0,0.003014,0.050362,0.004922,58195.0,0.004922,58195.0,-0.045078,55678.0,0.264235,56797.0,0.135135,50386.0
51322,adequate_choice_of_formulation,adequate choice of formulation,0.591206,1.700144e-06,1.0,0.0,1.0,0.002904,0.049524,0.005151,58194.0,0.005151,58194.0,-0.044849,55659.0,0.256012,57051.0,0.802817,7575.5
36160,quality_skill,quality skill,0.671589,1.700144e-06,0.0,2.0,2.0,0.006038,0.037163,0.005246,58193.0,0.005246,58193.0,-0.044754,55654.0,0.270790,56531.0,0.535211,24072.5
18562,physical_measurement,physical measurement,0.635023,1.133429e-06,0.0,1.0,1.0,0.003756,0.056023,0.005274,58192.0,0.005274,58192.0,-0.044726,55651.0,0.306664,54675.0,0.692308,14056.0
36168,electronics_field,electronics field,0.544002,1.700144e-06,0.0,3.0,3.0,0.007808,0.031050,0.005328,58191.0,0.005328,58191.0,-0.044672,55649.0,0.273869,56404.0,0.084507,52769.5
17410,breakdown_voltage,breakdown voltage,0.795428,1.983502e-06,0.0,1.0,1.0,0.002110,0.043472,0.005671,58190.0,0.005671,58190.0,-0.044329,55626.0,0.271069,56520.0,0.174603,48338.5
8157,validation_set,validation set,0.787044,1.133429e-06,0.0,1.0,1.0,0.002574,0.047010,0.005746,58189.0,0.005746,58189.0,-0.044254,55623.0,0.266775,56707.0,0.250000,44198.5


In [47]:
entities_df[entities_df['name'].str.contains('query')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
26288,live_queryable_decentralized_knowledge_graph,live queryable decentralized knowledge graph,0.395907,1.700144e-06,0.0,1.0,1.0,0.002016,0.042748,0.404611,914.0,0.404611,2641.0,0.354611,1542.0,0.800529,563.0,0.285714,42047.0
16229,count_distinct_sparql_query,count distinct sparql query,0.068806,5.667147e-07,2.0,2.0,4.0,0.016451,0.051689,0.299193,2103.0,0.299193,4973.0,0.249193,3149.0,0.569642,17194.0,0.833333,6235.0
3814,continuation_query,continuation query,0.561172,1.983502e-06,0.0,1.0,1.0,0.003581,0.054688,0.296106,2173.0,0.296106,5086.0,0.246106,3245.0,0.700447,4126.0,0.428571,32025.0
6629,hybrid_query_relaxation_solution,hybrid query relaxation solution,0.174377,1.700144e-06,2.0,0.0,2.0,0.010608,0.059152,0.289057,2316.0,0.289057,5358.0,0.239057,3426.0,0.628586,10009.0,0.857143,5017.0
52300,regular_sparql_query,regular sparql query,0.385354,1.700144e-06,0.0,1.0,1.0,0.002634,0.047464,0.264344,2870.0,0.264344,6327.0,0.214344,4158.0,0.702915,3979.0,0.444444,30758.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34227,query_q,query q,0.564783,1.133429e-06,0.0,1.0,1.0,0.002948,0.049861,0.037603,47759.0,0.037603,50758.0,-0.012397,46179.0,0.366327,49092.0,0.285714,42047.0
48276,query_execution_time,query execution time,0.496316,1.700144e-06,0.0,2.0,2.0,0.008005,0.046625,0.035872,48869.0,0.035872,51550.0,-0.014128,47052.0,0.361266,49667.0,0.470588,28789.0
27780,query_processing,query processing,0.537663,3.752320e-01,0.0,2.0,2.0,0.012592,0.068696,0.030201,52255.0,0.060402,40187.0,-0.039598,55211.0,0.314387,54145.0,0.627536,17899.0
6584,query_image,query image,0.477579,5.667147e-07,0.0,1.0,1.0,0.001741,0.040655,0.026234,54263.0,0.026234,55289.0,-0.023766,51329.0,0.418302,41562.0,0.133333,50478.0


# Visualize ranking families of entities

In [48]:
entities_df[entities_df['name'].str.contains('query')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
26288,live_queryable_decentralized_knowledge_graph,live queryable decentralized knowledge graph,0.395907,1.700144e-06,0.0,1.0,1.0,0.002016,0.042748,0.404611,914.0,0.404611,2641.0,0.354611,1542.0,0.800529,563.0,0.285714,42047.0
16229,count_distinct_sparql_query,count distinct sparql query,0.068806,5.667147e-07,2.0,2.0,4.0,0.016451,0.051689,0.299193,2103.0,0.299193,4973.0,0.249193,3149.0,0.569642,17194.0,0.833333,6235.0
3814,continuation_query,continuation query,0.561172,1.983502e-06,0.0,1.0,1.0,0.003581,0.054688,0.296106,2173.0,0.296106,5086.0,0.246106,3245.0,0.700447,4126.0,0.428571,32025.0
6629,hybrid_query_relaxation_solution,hybrid query relaxation solution,0.174377,1.700144e-06,2.0,0.0,2.0,0.010608,0.059152,0.289057,2316.0,0.289057,5358.0,0.239057,3426.0,0.628586,10009.0,0.857143,5017.0
52300,regular_sparql_query,regular sparql query,0.385354,1.700144e-06,0.0,1.0,1.0,0.002634,0.047464,0.264344,2870.0,0.264344,6327.0,0.214344,4158.0,0.702915,3979.0,0.444444,30758.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34227,query_q,query q,0.564783,1.133429e-06,0.0,1.0,1.0,0.002948,0.049861,0.037603,47759.0,0.037603,50758.0,-0.012397,46179.0,0.366327,49092.0,0.285714,42047.0
48276,query_execution_time,query execution time,0.496316,1.700144e-06,0.0,2.0,2.0,0.008005,0.046625,0.035872,48869.0,0.035872,51550.0,-0.014128,47052.0,0.361266,49667.0,0.470588,28789.0
27780,query_processing,query processing,0.537663,3.752320e-01,0.0,2.0,2.0,0.012592,0.068696,0.030201,52255.0,0.060402,40187.0,-0.039598,55211.0,0.314387,54145.0,0.627536,17899.0
6584,query_image,query image,0.477579,5.667147e-07,0.0,1.0,1.0,0.001741,0.040655,0.026234,54263.0,0.026234,55289.0,-0.023766,51329.0,0.418302,41562.0,0.133333,50478.0


In [49]:
entities_df[entities_df['name'].str.contains('cloud')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
19030,point_cloud_quality_assessment,point cloud quality assessment,0.329363,0.000002,1.0,1.0,2.0,0.008340,0.048240,0.414421,843.5,0.414421,2490.5,0.364421,1437.5,0.660481,6990.0,0.800000,7975.0
56514,explosion_of_demand_in_cloud_computing,explosion of demand in cloud computing,0.465210,0.000002,1.0,0.0,1.0,0.003474,0.053873,0.333841,1579.0,0.333841,3991.0,0.283841,2450.0,0.761012,1449.0,0.875000,4121.0
41952,cloud_related_architecture,cloud related architecture,0.264816,0.000001,2.0,3.0,5.0,0.016748,0.044779,0.307485,1943.0,0.307485,4692.0,0.257485,2942.0,0.615403,11478.0,0.000000,56570.5
26836,point_clouds_from_multiple_perspective,point clouds from multiple perspective,0.324709,0.000001,2.0,0.0,2.0,0.006969,0.041642,0.287008,2369.0,0.287008,5444.0,0.237008,3498.0,0.734719,2402.0,0.500000,26626.5
32247,edge_and_multi_cloud_environment,edge and multi cloud environment,0.357042,0.000002,0.0,1.0,1.0,0.004416,0.061056,0.269247,2762.0,0.269247,6117.0,0.219247,4019.0,0.593609,14043.0,0.166667,48764.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3217,cloud_domain,cloud domain,0.668690,0.000002,0.0,1.0,1.0,0.003012,0.050348,0.040501,45859.0,0.040501,49379.0,-0.009499,44659.0,0.409109,43054.0,0.473684,28601.5
2779,cloud_tier,cloud tier,0.623066,0.290503,0.0,2.0,2.0,0.009062,0.051712,0.037235,47992.0,0.074469,34468.0,-0.025531,51997.0,0.462434,34026.0,0.429375,31726.0
34379,cloud_modeling,cloud modeling,0.384796,0.000001,0.0,1.0,1.0,0.000865,0.033974,0.031679,51412.0,0.031679,53325.0,-0.018321,49056.0,0.403682,43926.0,0.500000,26626.5
47296,cloud_resource_allocation,cloud resource allocation,0.663933,0.000002,1.0,0.0,1.0,0.003720,0.055745,0.023197,55564.0,0.023197,56204.0,-0.026803,52439.0,0.544687,20764.0,0.435897,31443.0


In [50]:
entities_df[entities_df['name'].str.contains('federated')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
20675,fully_decentralized_hierarchical_federated_lea...,fully decentralized hierarchical federated lea...,0.275796,0.000002,0.0,1.0,1.0,0.003150,0.051398,0.252178,3235.0,0.252178,6925.0,0.202178,4608.0,0.822347,301.0,0.200000,46992.0
14908,modular_management_of_federated_learning_enhan...,modular management of federated learning enhan...,0.250181,0.000002,0.0,1.0,1.0,0.002070,0.043163,0.228253,4050.0,0.228253,8232.0,0.178253,5600.0,0.852046,114.0,0.523810,24864.5
13117,statistical_concerns_in_federated_learning,statistical concerns in federated learning,0.218752,0.000002,0.0,1.0,1.0,0.001028,0.035216,0.199507,5397.0,0.199507,10296.0,0.149507,7209.0,0.710306,3541.0,0.714286,12797.5
20697,classical_federated_learning_model,classical federated learning model,0.377624,0.000002,0.0,1.0,1.0,0.001599,0.039569,0.162109,8229.0,0.162109,14215.0,0.112109,10413.0,0.663879,6703.0,0.555556,22687.5
33054,federated_learning_with_aggregator,federated learning with aggregator,0.126546,0.000002,0.0,1.0,1.0,0.001362,0.037762,0.142955,10397.0,0.142955,16945.0,0.092955,12728.0,0.729944,2615.0,0.210526,46416.5
9976,federated_learning_based_network_intrusion_det...,federated learning based network intrusion det...,0.191243,0.000002,0.0,1.0,1.0,0.001282,0.037153,0.138353,11018.0,0.138353,17702.0,0.088353,13383.0,0.753467,1689.0,0.400000,34200.0
4250,fully_adaptive_federated_learning_system,fully adaptive federated learning system,0.319392,0.000002,0.0,1.0,1.0,0.005031,0.065739,0.133974,11651.0,0.133974,18474.0,0.083974,14036.0,0.789682,755.0,0.777778,9188.5
21948,federated_online_shopping_application,federated online shopping application,0.237490,0.000002,0.0,1.0,1.0,0.004742,0.063538,0.133041,11778.0,0.133041,18637.0,0.083041,14180.0,0.691666,4686.0,0.000000,56570.5
53109,license_aware_federated_query_processing_strategy,license aware federated query processing strategy,0.234991,0.000002,0.0,1.0,1.0,0.001891,0.041795,0.130737,12127.0,0.130737,19065.0,0.080737,14555.0,0.749516,1824.0,0.391304,34822.5
33055,federated_learning_with_election,federated learning with election,0.129985,0.000002,0.0,1.0,1.0,0.001362,0.037762,0.128308,12553.0,0.128308,19541.0,0.078308,14984.0,0.708114,3666.0,0.736842,11518.0


In [51]:
entities_df[entities_df['name'].str.contains('network')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
32683,network_analysis_tool,network analysis tool,0.465139,1.700144e-06,0.0,1.0,1.0,0.001951,0.042253,0.781429,57.0,0.781429,423.0,0.731429,165.0,0.478633,31271.0,0.500000,26626.5
40614,deep_neural_network_architecture_choice,deep neural network architecture choice,0.476929,1.700144e-06,0.0,1.0,1.0,0.000968,0.034759,0.633263,194.0,0.633263,865.0,0.583263,416.0,0.641133,8754.0,0.000000,56570.5
50891,anionic_three_dimensional_network,anionic three dimensional network,0.459090,1.700144e-06,2.0,0.0,2.0,0.009151,0.052138,0.557004,326.0,0.557004,1222.0,0.507004,630.0,0.825568,283.0,0.142857,50012.0
13809,multi_port_network_theory,multi port network theory,0.453682,1.700144e-06,0.0,1.0,1.0,0.002751,0.048357,0.539130,367.0,0.539130,1330.0,0.489130,698.0,0.575003,16422.0,0.666667,15619.0
11217,optimizing_mobile_network_design,optimizing mobile network design,0.627989,1.133429e-06,0.0,1.0,1.0,0.001961,0.042331,0.525719,403.0,0.525719,1403.0,0.475719,759.0,0.740220,2172.0,0.500000,26626.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51830,networking,networking,0.794188,5.667147e-07,0.0,1.0,1.0,0.000985,0.034889,0.016218,57416.0,0.016218,57558.0,-0.033782,54304.0,0.408351,43181.0,0.555556,22687.5
45068,three_dimensional_network,three dimensional network,0.687695,1.133429e-06,0.0,1.0,1.0,0.004654,0.062864,0.015427,57540.0,0.015427,57655.0,-0.034573,54460.0,0.373923,48176.0,0.678571,14752.5
2397,network_overhead,network overhead,0.537430,1.700144e-06,0.0,1.0,1.0,0.003578,0.054664,0.014641,57670.0,0.014641,57752.0,-0.035359,54611.0,0.307567,54627.0,0.733333,11691.5
36717,three_network_model,three network model,0.656983,1.700144e-06,8.0,1.0,9.0,0.057566,0.123039,0.014525,57688.0,0.014525,57769.0,-0.035475,54640.0,0.410618,42826.0,0.648649,16776.5


In [52]:
entities_df[entities_df['name'].str.contains('trajectory')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
49196,different_points_of_the_trajectory,different points of the trajectory,0.658858,0.000002,0.0,1.0,1.0,0.001964,0.042354,0.436941,723.0,0.436941,2188.0,0.386941,1258.0,0.713087,3383.0,0.500000,26626.5
2667,sets_of_trajectory,sets of trajectory,0.448852,0.424620,0.0,2.0,2.0,0.009081,0.051803,0.342122,1475.0,0.684244,684.0,0.584244,414.0,0.443861,37245.0,0.403846,33725.0
21227,baseline_trajectory,baseline trajectory,0.482230,0.000002,1.0,0.0,1.0,0.001055,0.035426,0.253129,3197.0,0.253129,6866.0,0.203129,4564.0,0.490146,29398.0,0.000000,56570.5
15983,first_trajectory_planning_problem,first trajectory planning problem,0.404361,0.000002,1.0,0.0,1.0,0.001218,0.036670,0.225310,4183.0,0.225310,8436.0,0.175310,5757.0,0.819300,334.0,0.428571,32025.0
29651,unmanned_aerial_vehicle_trajectory,unmanned aerial vehicle trajectory,0.400937,0.000002,1.0,0.0,1.0,0.002354,0.045328,0.218264,4475.0,0.218264,8905.0,0.168264,6112.0,0.648881,8019.0,0.000000,56570.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46912,linear_trajectory,linear trajectory,0.690089,0.000002,1.0,1.0,2.0,0.010238,0.057371,0.025643,54539.0,0.025643,55481.0,-0.024357,51556.0,0.307577,54626.0,0.652174,16597.0
15975,trajectory_between_two_coordinate,trajectory between two coordinate,0.462867,0.000002,0.0,1.0,1.0,0.001516,0.038936,0.024576,54994.0,0.024576,55804.0,-0.025424,51953.0,0.330902,52826.0,0.821429,6860.0
20394,system_trajectory,system trajectory,0.577844,0.000002,1.0,1.0,2.0,0.010650,0.059354,0.024449,55047.0,0.024449,55846.0,-0.025551,52004.0,0.295706,55330.0,0.882353,3743.0
9192,dynamic_trajectory,dynamic trajectory,0.462277,0.000002,0.0,2.0,2.0,0.010398,0.058141,0.016985,57255.0,0.016985,57429.0,-0.033015,54137.0,0.247584,57250.0,0.473684,28601.5


In [53]:
entities_df[entities_df['name'].str.contains('sparql')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
3792,sparql_federation_engine,sparql federation engine,0.230511,2.372682e-01,2.0,2.0,4.0,0.026964,0.086214,0.361398,1266.0,0.722796,578.0,0.622796,315.0,0.564268,17904.0,0.083333,52780.0
16229,count_distinct_sparql_query,count distinct sparql query,0.068806,5.667147e-07,2.0,2.0,4.0,0.016451,0.051689,0.299193,2103.0,0.299193,4973.0,0.249193,3149.0,0.569642,17194.0,0.833333,6235.0
52300,regular_sparql_query,regular sparql query,0.385354,1.700144e-06,0.0,1.0,1.0,0.002634,0.047464,0.264344,2870.0,0.264344,6327.0,0.214344,4158.0,0.702915,3979.0,0.444444,30758.0
54010,decentralized_sparql_query_processing,decentralized sparql query processing,0.227607,1.133429e-06,2.0,0.0,2.0,0.007374,0.043592,0.251437,3261.0,0.251437,6964.0,0.201437,4643.0,0.753428,1690.0,0.466667,29092.5
3812,sparql_query_engine,sparql query engine,0.363612,3.373602e-01,0.0,2.0,2.0,0.010079,0.056607,0.213382,4718.0,0.426764,2332.0,0.326764,1847.0,0.661352,6914.0,0.471429,28697.5
48256,wikidata_sparql_endpoint,wikidata sparql endpoint,0.400160,1.700144e-06,1.0,0.0,1.0,0.003834,0.056618,0.198112,5476.0,0.198112,10400.0,0.148112,7299.0,0.758242,1537.0,0.857143,5017.0
52992,public_sparql_query_server,public sparql query server,0.287860,1.700144e-06,1.0,0.0,1.0,0.001448,0.038420,0.174780,7086.0,0.174780,12712.0,0.124780,9159.0,0.655409,7415.0,0.200000,46992.0
31869,complex_sparql_property_path_query,complex sparql property path query,0.156291,2.647714e-01,5.0,3.0,8.0,0.047092,0.104931,0.163744,8052.0,0.654974,784.0,0.454974,862.0,0.676951,5699.0,0.684261,14454.0
10515,sparql_continuation_query,sparql continuation query,0.266895,1.983502e-06,0.0,1.0,1.0,0.002570,0.046977,0.152823,9183.0,0.152823,15448.0,0.102823,11452.0,0.592780,14144.0,0.900000,2848.5
15742,large_federations_of_sparql_endpoint,large federations of sparql endpoint,0.265893,2.540990e-01,3.0,1.0,4.0,0.015032,0.047028,0.151951,9268.0,0.455854,1982.0,0.305854,2111.0,0.646998,8200.0,0.654401,16524.0


In [54]:
entities_df[entities_df['name'].str.contains('large')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
50585,large_number_of_point,large number of point,0.496099,2.266859e-06,0.0,1.0,1.0,0.001255,0.036950,0.672552,139.0,0.672552,715.0,0.622552,316.0,0.530608,22914.0,0.500000,26626.5
14879,large_amounts_of_training_datum,large amounts of training datum,0.524993,1.700144e-06,0.0,1.0,1.0,0.002161,0.043856,0.622561,210.0,0.622561,906.0,0.572561,441.0,0.540640,21414.0,0.666667,15619.0
57474,large_scale_finite_element_analysis,large scale finite element analysis,0.445491,1.700144e-06,0.0,1.0,1.0,0.003017,0.050384,0.584334,269.0,0.584334,1065.0,0.534334,527.0,0.685828,5068.0,0.750000,10736.0
50213,silicon_nanowire_with_larger_diameter,silicon nanowire with larger diameter,0.271955,5.667147e-07,2.0,0.0,2.0,0.011131,0.061665,0.383590,1065.0,0.383590,2967.0,0.333590,1751.0,0.613285,11701.0,0.750000,10736.0
17612,large_and_positive_effect,large and positive effect,0.509450,1.700144e-06,0.0,3.0,3.0,0.020821,0.080665,0.360434,1277.0,0.360434,3372.0,0.310434,2056.0,0.521445,24346.0,0.750000,10736.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17398,very_large_scale_integration,very large scale integration,0.574359,1.983502e-06,1.0,0.0,1.0,0.002337,0.045200,0.012968,57870.0,0.012968,57905.0,-0.037032,54873.0,0.459576,34541.0,0.936508,1145.0
17392,large_area_component,large area component,0.690866,1.983502e-06,3.0,0.0,3.0,0.018592,0.072166,0.011307,58008.0,0.011307,58024.0,-0.038693,55108.0,0.425768,40299.0,0.650794,16638.5
36718,large_strain_value,large strain value,0.484234,1.700144e-06,0.0,1.0,1.0,0.001456,0.038481,0.011215,58012.0,0.011215,58028.0,-0.038785,55121.0,0.349765,51038.0,0.135135,50386.0
36151,large_number_of_object,large number of object,0.652419,1.700144e-06,1.0,1.0,2.0,0.008745,0.050188,0.010859,58034.0,0.010859,58049.0,-0.039141,55167.0,0.434598,38825.0,0.690141,14202.5


In [55]:
entities_df[entities_df['name'].str.contains('model')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
55481,modeling_the_remediation_of_organic_contaminant,modeling the remediation of organic contaminant,0.344924,0.000002,0.0,1.0,1.0,0.001510,0.038895,0.936762,1.0,0.936762,220.0,0.886762,52.0,0.712075,3434.0,0.500000,26626.5
31558,pre_trained_language_model_camembert,pre trained language model camembert,0.382875,0.000002,0.0,1.0,1.0,0.001627,0.039783,0.869472,6.0,0.869472,280.0,0.819472,74.0,0.669174,6300.0,0.500000,26626.5
10967,hierarchical_bayesian_model,hierarchical bayesian model,0.318415,0.000001,1.0,0.0,1.0,0.003680,0.055438,0.854552,11.0,0.854552,296.0,0.804552,90.0,0.652680,7668.0,0.500000,26626.5
53080,modeling_effort,modeling effort,0.669345,0.000002,1.0,0.0,1.0,0.001013,0.035105,0.818386,34.0,0.818386,356.0,0.768386,126.0,0.554314,19279.0,0.500000,26626.5
25381,designed_filter_model,designed filter model,0.440269,0.000001,0.0,1.0,1.0,0.004141,0.058955,0.718876,102.0,0.718876,589.0,0.668876,240.0,0.489679,29483.0,0.500000,26626.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10765,human_model,human model,0.824441,0.000001,1.0,0.0,1.0,0.003493,0.054013,0.013140,57855.0,0.013140,57895.0,-0.036860,54849.0,0.309388,54484.0,0.090909,52540.0
57987,obtained_model,obtained model,0.539330,0.000002,1.0,2.0,3.0,0.013563,0.052992,0.010738,58040.0,0.010738,58053.0,-0.039262,55179.0,0.281554,56059.0,0.675676,14821.0
33436,dynamics_model_constraint,dynamics model constraint,0.577768,0.000001,1.0,0.0,1.0,0.003727,0.055796,0.009395,58106.0,0.009395,58113.0,-0.040605,55318.0,0.372216,48373.0,0.907692,2551.0
33433,evaluation_model,evaluation model,0.663805,0.000001,2.0,0.0,2.0,0.005966,0.036818,0.008694,58132.0,0.008694,58136.0,-0.041306,55382.0,0.354296,50521.0,0.784615,8859.5


In [56]:
entities_df[entities_df['name'].str.contains('method')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
49052,single_scan_method,single scan method,0.303337,2.266859e-06,1.0,0.0,1.0,0.003009,0.050321,0.723804,100.0,0.723804,573.0,0.673804,236.0,0.555990,19044.0,0.500000,26626.5
19150,micro_phenomenological_interview_method,micro phenomenological interview method,0.485067,1.700144e-06,0.0,1.0,1.0,0.002449,0.046051,0.604932,237.0,0.604932,982.0,0.554932,482.0,0.715142,3287.0,0.666667,15619.0
23837,advanced_hybrid_aerosol_pecvd_method,advanced hybrid aerosol pecvd method,0.374968,1.700144e-06,2.0,0.0,2.0,0.010419,0.058242,0.566597,302.0,0.566597,1155.0,0.516597,588.0,0.674697,5864.0,0.666667,15619.0
52378,surface_tension_measurement_method,surface tension measurement method,0.254091,1.700144e-06,0.0,1.0,1.0,0.002374,0.045479,0.544976,354.0,0.544976,1296.0,0.494976,681.0,0.551831,19669.0,0.666667,15619.0
13754,precise_methodological_tool,precise methodological tool,0.426355,1.133429e-06,1.0,0.0,1.0,0.002761,0.048436,0.491400,496.0,0.491400,1641.0,0.441400,921.0,0.328649,53033.0,0.000000,56570.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48573,sampling_method,sampling method,0.586378,1.133429e-06,3.0,0.0,3.0,0.013830,0.054008,0.011316,58007.0,0.011316,58023.0,-0.038684,55106.0,0.338201,52175.0,0.378378,35638.0
18548,spectral_method,spectral method,0.530699,1.133429e-06,0.0,1.0,1.0,0.001585,0.039462,0.009870,58090.0,0.009870,58099.0,-0.040130,55275.0,0.452875,35708.0,0.756410,10185.0
51924,efficient_methodology,efficient methodology,0.600000,1.133429e-06,1.0,1.0,2.0,0.008802,0.050463,0.008862,58128.0,0.008862,58132.0,-0.041138,55371.0,0.220183,57766.0,0.592593,20299.5
8146,temporal_interpolation_method,temporal interpolation method,0.667880,1.133429e-06,1.0,2.0,3.0,0.013984,0.054596,0.008858,58129.0,0.008858,58133.0,-0.041142,55372.0,0.359342,49885.0,0.696429,13824.0


In [57]:
entities_df[entities_df['name'].str.contains('food')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
24499,hygiene_culture_of_agri_food_worker,hygiene culture of agri food worker,0.309986,0.000002,1.0,0.0,1.0,0.001305,0.037327,0.523158,416.0,0.523158,1425.0,0.473158,775.0,0.727431,2704.0,0.000000,56570.5
3573,food_authentication,food authentication,0.328690,0.307819,2.0,1.0,3.0,0.017069,0.066361,0.458405,617.0,0.916809,234.0,0.816809,79.0,0.594029,13990.0,0.375000,35966.0
32301,agriculture_and_food_processing_waste_valoriza...,agriculture and food processing waste valoriza...,0.230748,0.000002,1.0,0.0,1.0,0.003058,0.050700,0.454929,647.0,0.454929,1992.0,0.404929,1140.0,0.666053,6540.0,0.750000,10736.0
8625,heavy_metal_contamination_in_food_product,heavy metal contamination in food product,0.263353,0.000002,0.0,2.0,2.0,0.009416,0.053417,0.381890,1080.0,0.381890,2999.0,0.331890,1775.0,0.736270,2334.0,0.285714,42047.0
28083,refrigerated_warehousing_of_food,refrigerated warehousing of food,0.433797,0.000002,0.0,1.0,1.0,0.001965,0.042363,0.343794,1453.0,0.343794,3726.0,0.293794,2297.0,0.847501,138.0,0.611111,19026.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9113,food_transition,food transition,0.578629,0.000001,0.0,1.0,1.0,0.000661,0.032417,0.021927,55994.0,0.021927,56508.0,-0.028073,52844.0,0.565635,17696.0,0.666667,15619.0
3118,food_compositional_datum,food compositional datum,0.473630,0.000002,0.0,1.0,1.0,0.003639,0.055131,0.021142,56213.0,0.021142,56677.0,-0.028858,53063.0,0.423366,40734.0,0.633333,17650.0
9110,sustainable_food,sustainable food,0.597329,0.000001,0.0,1.0,1.0,0.001224,0.036711,0.020884,56310.0,0.020884,56746.0,-0.029116,53151.0,0.553626,19375.0,0.022222,54927.0
19870,food_residue,food residue,0.768243,0.000002,1.0,0.0,1.0,0.001788,0.041010,0.014789,57648.0,0.014789,57735.0,-0.035211,54583.0,0.425029,40436.0,0.769231,9649.0


In [58]:
entities_df[entities_df['name'].str.contains('food waste')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
15705,no_food_waste_practice,no food waste practice,0.572441,0.000003,3.0,1.0,4.0,0.015103,0.047262,0.094924,19910.0,0.094924,27599.0,0.044924,22301.0,0.634930,9355.0,0.500000,26626.5
23607,agri_food_wastewater_treatment,agri food wastewater treatment,0.340299,0.000002,0.0,1.0,1.0,0.001541,0.039127,0.086386,22557.0,0.086386,30230.0,0.036386,24838.0,0.677359,5669.0,0.000000,56570.5
24791,global_food_waste,global food waste,0.593232,0.000003,0.0,1.0,1.0,0.001543,0.039145,0.079058,25198.0,0.079058,32760.0,0.029058,27312.0,0.610376,12025.0,0.384615,35273.0
37982,drying_of_food_waste,drying of food waste,0.442557,0.000002,4.0,0.0,4.0,0.014848,0.046423,0.058641,34739.0,0.058641,40968.0,0.008641,35622.0,0.529932,23025.0,0.388889,34968.0
37992,food_waste,food waste,0.649087,0.454112,0.0,2.0,2.0,0.008016,0.046679,0.041057,45486.0,0.082115,31657.0,-0.017885,48842.0,0.443086,37366.0,0.878472,3868.0


In [59]:
entities_df[entities_df['name'].str.contains('virtual reality')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
19174,virtual_reality_immersive_soundscape,virtual reality immersive soundscape,0.297063,1.700144e-06,0.0,1.0,1.0,0.000731,0.032951,0.402346,929.0,0.402346,2671.0,0.352346,1564.0,0.771263,1133.0,0.888889,3409.5
5363,virtual_reality_hmd,virtual reality hmd,0.407573,2.266859e-06,1.0,0.0,1.0,0.002111,0.043473,0.251727,3251.0,0.251727,6946.0,0.201727,4628.0,0.765577,1318.0,0.666667,15619.0
31807,head_mounted_virtual_reality_device,head mounted virtual reality device,0.398984,1.700144e-06,1.0,0.0,1.0,0.002532,0.046683,0.229525,3991.0,0.229525,8142.0,0.179525,5533.0,0.705577,3808.0,0.777778,9188.5
11880,virtual_reality_based_om_test_protocol,virtual reality based om test protocol,0.270367,1.133429e-06,2.0,0.0,2.0,0.012830,0.069842,0.204430,5156.0,0.204430,9912.0,0.154430,6923.0,0.672579,6021.0,0.583333,20797.0
7579,social_virtual_reality_platform,social virtual reality platform,0.292720,3.074615e-01,1.0,1.0,2.0,0.007844,0.045854,0.186391,6228.0,0.372783,3129.0,0.272783,2631.0,0.725006,2806.0,0.468518,28926.0
38496,virtual_reality_tool,virtual reality tool,0.356127,1.133429e-06,1.0,0.0,1.0,0.003531,0.054302,0.185676,6286.0,0.185676,11563.0,0.135676,8240.0,0.492045,29079.0,0.000000,56570.5
11863,virtual_reality_accessibility,virtual reality accessibility,0.273141,1.983502e-06,0.0,1.0,1.0,0.000357,0.030098,0.184226,6382.0,0.184226,11704.0,0.134226,8349.0,0.651308,7795.0,0.500000,26626.5
43225,fully_embodied_virtual_reality_exposure_therapy,fully embodied virtual reality exposure therapy,0.229263,1.133429e-06,1.0,1.0,2.0,0.007586,0.044612,0.170272,7446.0,0.170272,13202.0,0.120272,9564.0,0.754536,1648.0,0.909091,2405.5
56054,virtual_reality_immersion,virtual reality immersion,0.351619,1.983502e-06,2.0,1.0,3.0,0.013752,0.053714,0.167425,7717.0,0.167425,13552.0,0.117425,9858.0,0.494426,28720.0,0.750000,10736.0
12479,virtual_reality_exposure_therapy,virtual reality exposure therapy,0.265130,2.932397e-01,1.0,2.0,3.0,0.016959,0.065940,0.166282,7826.0,0.332563,4014.0,0.232563,3624.0,0.593236,14090.0,0.253247,43697.0


In [ ]:
from src.commands.build_emap.clustering import CommunityDetectionBasedClustering
from src.commands.build_emap.representative_nodes_selection import RepresentativeEntitiesSelector

In [65]:
from src.helpers.torch_utils import create_similarity_graph
from sknetwork.clustering import Leiden
from scipy.sparse import csr_matrix
from collections import Counter

In [71]:
X = results['baseline_2_final']['entities_sem_embeddings']
X = X / X.norm(dim=-1, p=2, keepdim=True)
edge_index, edge_weight = create_similarity_graph(x=X, th=0.5, batch_size=1024)

In [72]:
def leiden_clustering(
    edge_index: torch.Tensor,
    edge_weight: torch.Tensor,
    num_nodes: int,
    resolution: float = 1.0,
    random_state: int = 42,
):

    edge_index = edge_index.detach().cpu().numpy()
    edge_weight = edge_weight.detach().cpu().numpy()

    # Build sparse adjacency matrix
    adjacency = csr_matrix(
        (edge_weight, (edge_index[0], edge_index[1])),
        shape=(num_nodes, num_nodes),
        dtype=np.float32
    )

    # Make graph undirected
    adjacency = adjacency.maximum(adjacency.T)
    adjacency.setdiag(0)
    adjacency.eliminate_zeros()

    # Leiden
    leiden = Leiden(
        resolution=resolution,
        modularity="newman",
        random_state=random_state,
        return_probs=False,
        return_aggregate=False
    )

    labels = leiden.fit_predict(adjacency)

    return labels, leiden

In [73]:
labels, model = leiden_clustering(
    edge_index=edge_index,
    edge_weight=edge_weight,
    num_nodes=X.size(0),
    resolution=1,
    random_state=42
)

In [75]:
uniques, counts = np.unique(labels, return_counts=True)

In [78]:
entities_df.iloc[labels == 0].sort_values(by='avg_Score', ascending=False).head(10)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
13754,precise_methodological_tool,precise methodological tool,0.426355,1.133429e-06,1.0,0.0,1.0,0.002761,0.048436,0.491400,496.0,0.491400,1641.0,0.441400,921.0,0.328649,53033.0,0.000000,56570.5
16835,approaches_to_handle_disruption,approaches to handle disruption,0.236069,1.983502e-06,1.0,0.0,1.0,0.001483,0.038686,0.470712,569.0,0.470712,1832.0,0.420712,1043.0,0.647317,8167.0,0.800000,7975.0
48166,global_optimisation_problem,global optimisation problem,0.446385,1.133429e-06,0.0,2.0,2.0,0.008981,0.051323,0.417368,830.0,0.417368,2452.0,0.367368,1412.0,0.441832,37582.0,0.333333,38943.0
34297,interactive_microbial_genetic_algorithm,interactive microbial genetic algorithm,0.371337,1.700144e-06,2.0,0.0,2.0,0.011824,0.065001,0.299967,2090.0,0.299967,4943.0,0.249967,3136.0,0.703822,3916.0,0.000000,56570.5
41353,risk_neutral_optimization_algorithm,risk neutral optimization algorithm,0.297824,1.700144e-06,0.0,1.0,1.0,0.004779,0.063819,0.291736,2269.0,0.291736,5258.0,0.241736,3364.0,0.661999,6859.0,0.333333,38943.0
10338,timely_assessing_the_disruption_risk,timely assessing the disruption risk,0.388934,5.667147e-07,1.0,0.0,1.0,0.003923,0.057293,0.289646,2303.0,0.289646,5331.0,0.239646,3409.0,0.745230,1974.0,0.111111,51598.0
56309,metallurgical_analysis,metallurgical analysis,0.618247,2.266859e-06,1.0,0.0,1.0,0.002654,0.047619,0.285365,2404.0,0.285365,5509.0,0.235365,3556.0,0.398467,44742.0,0.250000,44198.5
37272,genetic_programming_concept,genetic programming concept,0.365712,1.163987e-01,0.0,1.0,1.0,0.001523,0.038993,0.266405,2822.0,0.532811,1362.0,0.432811,966.0,0.735768,2358.0,0.715278,12522.5
46537,discrete_stochastic_constraint,discrete stochastic constraint,0.348049,2.348222e-01,0.0,2.0,2.0,0.009879,0.055642,0.261260,2944.0,0.522519,1427.0,0.422519,1028.0,0.507803,26517.0,0.350000,37789.5
29647,accurate_quantification_of_molecule,accurate quantification of molecule,0.378278,1.133429e-06,0.0,1.0,1.0,0.003026,0.050451,0.261046,2953.0,0.261046,6471.0,0.211046,4261.0,0.455427,35268.0,0.800000,7975.0


In [ ]:
import math


0.22944099999999998

In [131]:
idx = 0

mask = RepresentativeEntitiesSelector(
    th_eliminate=0.3,
    th_merge=0.5,
    k=15
).select_representative_nodes(TensorDict({
    "x" : X[labels == idx,:],
    "distances" : X[labels == idx,:] @ X[labels == idx,:].mean(dim=0, keepdim=True).t()
}))

mask.sum()

tensor(476)

In [104]:
mask.argwhere()

tensor([[ 2],
        [ 4],
        [11]])

In [124]:
(labels == 50).sum()

np.int64(271)

In [87]:
mask.sum()

tensor(515)

In [108]:
entities_df.iloc[labels == 100]['name'].tolist()

['coverability problem',
 'coverability',
 'set covering problem',
 'set covering component',
 'partitioning process',
 'initial partitioning',
 'arbitrary balanced partitioning',
 'partition',
 'maximal covering location problem',
 'edge set partitioning problem',
 'multi type tree partitioning decision',
 'set partitioning formulation',
 'partitioning',
 'set partitioning problem',
 'optimal partition',
 'multi type tree partitioning scheme',
 'set partitioning component',
 'set cover problem']

In [106]:
entities_df.iloc[labels == 100].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
45487,edge_set_partitioning_problem,edge set partitioning problem,0.222787,2.266859e-06,1.0,1.0,2.0,0.009938,0.055925,0.404373,917.0,0.404373,2646.0,0.354373,1547.0,0.644993,8394.0,0.800000,7975.0
48240,multi_type_tree_partitioning_decision,multi type tree partitioning decision,0.508110,2.266859e-06,1.0,0.0,1.0,0.001650,0.039961,0.276176,2580.0,0.276176,5828.0,0.226176,3784.0,0.712132,3431.0,0.714286,12797.5
39525,maximal_covering_location_problem,maximal covering location problem,0.192605,2.064417e-01,7.0,0.0,7.0,0.045004,0.106973,0.167717,7691.0,0.335434,3941.0,0.235434,3555.0,0.660771,6956.0,0.714286,12797.5
34589,arbitrary_balanced_partitioning,arbitrary balanced partitioning,0.524783,1.133429e-06,0.0,2.0,2.0,0.010437,0.058327,0.155479,8901.0,0.155479,15078.0,0.105479,11142.0,0.648512,8057.0,0.083333,52905.5
53322,multi_type_tree_partitioning_scheme,multi type tree partitioning scheme,0.395329,1.855455e-01,0.0,2.0,2.0,0.009471,0.053680,0.133348,11735.0,0.266697,6230.0,0.166697,6198.0,0.610864,11976.0,0.314286,40228.0
55284,set_partitioning_component,set partitioning component,0.557230,1.133429e-06,1.0,0.0,1.0,0.004510,0.061771,0.119290,14177.0,0.119290,21416.0,0.069290,16655.0,0.581864,15504.0,0.846154,5578.5
48838,set_partitioning_formulation,set partitioning formulation,0.544161,1.700144e-06,2.0,0.0,2.0,0.012134,0.066494,0.117973,14438.0,0.117973,21721.0,0.067973,16925.0,0.599987,13253.0,0.733333,11691.5
24614,partitioning_process,partitioning process,0.589836,5.667147e-07,1.0,0.0,1.0,0.004605,0.062498,0.104963,17292.0,0.104963,24828.0,0.054963,19764.0,0.616586,11350.0,0.714286,12797.5
51578,optimal_partition,optimal partition,0.404763,1.700144e-06,2.0,0.0,2.0,0.010811,0.060129,0.103849,17566.0,0.103849,25126.0,0.053849,20037.0,0.461352,34222.0,0.416667,32887.5
57091,set_cover_problem,set cover problem,0.426875,1.700144e-06,0.0,1.0,1.0,0.002547,0.046798,0.098655,18833.0,0.098655,26485.0,0.048655,21258.0,0.591874,14263.0,0.875000,4121.0


In [ ]:
entities_df.iloc[labels == 2].sort_values(by='avg_Score', ascending=False).head(15)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
25561,thiazole_ring,thiazole ring,0.576707,1.700144e-06,0.0,1.0,1.0,0.002218,0.044291,0.821290,30.0,0.821290,348.0,0.771290,122.0,0.558597,18676.0,0.500000,26626.5
26688,fast_charging_energy_storage_device,fast charging energy storage device,0.458745,2.266859e-06,0.0,1.0,1.0,0.001101,0.035776,0.795973,45.0,0.795973,389.0,0.745973,145.0,0.694670,4498.0,0.500000,26626.5
20203,excited_state_proton_transfer_phenomena,excited state proton transfer phenomena,0.255840,1.700144e-06,0.0,1.0,1.0,0.001082,0.035626,0.760854,68.0,0.760854,474.0,0.710854,186.0,0.630048,9846.0,0.500000,26626.5
1874,delay_in_the_yield,delay in the yield,0.530521,1.700144e-06,0.0,1.0,1.0,0.001524,0.038997,0.736479,95.0,0.736479,541.0,0.686479,222.0,0.558799,18642.0,0.500000,26626.5
56153,major_charge_carrier,major charge carrier,0.645595,1.700144e-06,1.0,0.0,1.0,0.001793,0.041050,0.691538,123.0,0.691538,654.0,0.641538,278.0,0.622864,10640.0,0.500000,26626.5
49373,protonated_iminothioindoxyl,protonated iminothioindoxyl,0.463934,1.700144e-06,1.0,0.0,1.0,0.002831,0.048966,0.644019,177.0,0.644019,823.0,0.594019,383.0,0.782668,899.0,0.750000,10736.0
49084,opto_electronic_property,opto electronic property,0.429394,1.133429e-06,1.0,1.0,2.0,0.006478,0.039281,0.604562,238.0,0.604562,984.0,0.554562,483.0,0.578883,15892.0,0.666667,15619.0
56067,relevant_excited_state,relevant excited state,0.619169,1.133429e-06,0.0,1.0,1.0,0.001353,0.037697,0.588284,263.0,0.588284,1047.0,0.538284,516.0,0.466686,33304.0,0.500000,26626.5
50632,photoelectrochemical_property,photoelectrochemical property,0.313670,1.700144e-06,1.0,1.0,2.0,0.008751,0.050214,0.571005,295.0,0.571005,1130.0,0.521005,574.0,0.624370,10451.0,0.666667,15619.0
41970,anatase_phase_titania_surface,anatase phase titania surface,0.243713,1.700144e-06,2.0,0.0,2.0,0.010122,0.056813,0.569628,297.0,0.569628,1136.0,0.519628,577.0,0.742102,2096.0,0.666667,15619.0


In [ ]:
entities_df.iloc[labels == 3].sort_values(by='avg_Score', ascending=False).head(15)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
18032,forward_kinematic_model_of_the_instrument,forward kinematic model of the instrument,0.384889,1.133429e-06,1.0,0.0,1.0,0.003227,0.051987,0.695578,121.0,0.695578,643.0,0.645578,274.0,0.731557,2543.0,0.750000,10736.0
20352,inverse_dynamics_model,inverse dynamics model,0.443366,1.700144e-06,1.0,0.0,1.0,0.001665,0.040078,0.641943,181.0,0.641943,832.0,0.591943,388.0,0.544386,20810.0,0.500000,26626.5
36320,autonomous_vehicle_maneuvering_security,autonomous vehicle maneuvering security,0.461951,1.133429e-06,0.0,1.0,1.0,0.001755,0.040763,0.630795,199.0,0.630795,876.0,0.580795,423.0,0.762050,1418.0,0.000000,56570.5
40444,control_design_of_a_robot,control design of a robot,0.206494,1.700144e-06,1.0,0.0,1.0,0.000875,0.034047,0.616764,218.0,0.616764,933.0,0.566764,452.0,0.483844,30422.0,0.666667,15619.0
27299,gripper_activated_by_mckibbens_pneumatic_muscle,gripper activated by mckibbens pneumatic muscle,0.177939,1.133429e-06,0.0,1.0,1.0,0.002536,0.046716,0.545652,352.0,0.545652,1290.0,0.495652,678.0,0.642553,8629.0,0.666667,15619.0
42858,manipulators_with_tensegrity_x_joint,manipulators with tensegrity x joint,0.165298,8.502932e-02,3.0,0.0,3.0,0.024539,0.094843,0.525296,407.0,1.050591,146.0,0.950591,40.0,0.794185,679.0,0.458333,29680.0
9621,reconfigurable_autonomous_underwater_robot_con...,reconfigurable autonomous underwater robot con...,0.209397,1.700144e-06,0.0,2.0,2.0,0.009897,0.055731,0.479617,534.0,0.479617,1747.0,0.429617,985.0,0.758772,1524.0,0.750000,10736.0
31726,sagging_cable_driven_parallel_robot,sagging cable driven parallel robot,0.255561,1.700144e-06,0.0,1.0,1.0,0.003525,0.054258,0.472466,562.0,0.472466,1813.0,0.422466,1029.0,0.671922,6064.0,0.750000,10736.0
53938,continuum_planar_parallel_robot,continuum planar parallel robot,0.199520,5.667147e-07,0.0,1.0,1.0,0.005008,0.065567,0.467487,583.0,0.467487,1859.0,0.417487,1058.0,0.739720,2203.0,0.142857,50012.0
53955,planar_robotic_manipulator,planar robotic manipulator,0.460257,1.700144e-06,0.0,1.0,1.0,0.001511,0.038904,0.459246,614.0,0.459246,1937.0,0.409246,1106.0,0.640245,8847.0,0.833333,6235.0


In [ ]:
entities_df.iloc[labels == 4].sort_values(by='avg_Score', ascending=False).head(15)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
50139,organic_based_battery_electrode_material,organic based battery electrode material,0.154265,0.000002,0.0,1.0,1.0,0.002593,0.047154,0.833264,24.0,0.833264,329.0,0.783264,110.0,0.743538,2049.0,0.500000,26626.5
39127,electrolyte_behavior,electrolyte behavior,0.413263,0.000001,1.0,0.0,1.0,0.001271,0.037069,0.793151,50.0,0.793151,398.0,0.743151,150.0,0.466290,33369.0,0.500000,26626.5
10657,solid_solution_type_storage_mechanism,solid solution type storage mechanism,0.454154,0.000002,0.0,1.0,1.0,0.003050,0.050635,0.792613,51.0,0.792613,399.0,0.742613,153.0,0.753244,1698.0,0.500000,26626.5
9117,resistivity_variation,resistivity variation,0.282956,0.000002,0.0,1.0,1.0,0.001917,0.041999,0.754219,75.0,0.754219,493.0,0.704219,194.0,0.491935,29092.0,0.500000,26626.5
44817,organic_semiconductor_based_electronic_device,organic semiconductor based electronic device,0.425632,0.000002,2.0,0.0,2.0,0.008612,0.049548,0.681759,132.0,0.681759,687.0,0.631759,301.0,0.747512,1890.0,0.750000,10736.0
12656,solid_state_ionogel_electrolyte,solid state ionogel electrolyte,0.354179,0.000002,0.0,1.0,1.0,0.001752,0.040741,0.568810,299.0,0.568810,1143.0,0.518810,580.0,0.625739,10318.0,0.333333,38943.0
9543,electrochemical_measurement_system,electrochemical measurement system,0.371910,0.000001,1.0,0.0,1.0,0.003330,0.052768,0.517443,435.0,0.517443,1462.0,0.467443,805.0,0.692540,4637.0,0.666667,15619.0
24360,organic_field_effect_transistor,organic field effect transistor,0.299560,0.176927,2.0,0.0,2.0,0.014630,0.078504,0.436419,725.0,0.872838,275.0,0.772838,119.0,0.680473,5458.0,0.305556,40807.0
8653,uv_vis_nir_spectroelectrochemical_investigation,uv vis nir spectroelectrochemical investigation,0.593343,0.000002,1.0,0.0,1.0,0.003359,0.052993,0.426470,780.0,0.426470,2335.0,0.376470,1337.0,0.667041,6444.0,0.750000,10736.0
18810,packed_bed_thermal_energy_storage_system,packed bed thermal energy storage system,0.299852,0.309589,3.0,1.0,4.0,0.018730,0.059173,0.390966,1014.0,0.781933,420.0,0.681933,226.0,0.719622,3071.0,0.453947,30173.0


In [ ]:
entities_df.iloc[labels == 5].sort_values(by='avg_Score', ascending=False).head(15)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
42536,general_operator,general operator,0.633344,0.000002,0.0,1.0,1.0,0.002576,0.047024,0.550400,343.0,0.550400,1262.0,0.500400,657.0,0.383374,46943.0,0.500000,26626.5
37815,energy_flows_between_component,energy flows between component,0.611875,0.000001,0.0,1.0,1.0,0.003436,0.053583,0.536131,373.0,0.536131,1346.0,0.486131,709.0,0.688602,4871.0,0.600000,19752.0
45868,excessive_volatility,excessive volatility,0.316489,0.000001,1.0,0.0,1.0,0.001458,0.038498,0.483202,523.0,0.483202,1712.0,0.433202,965.0,0.641607,8713.0,0.666667,15619.0
50180,main_items_of_energy_saving,main items of energy saving,0.345753,0.000001,0.0,1.0,1.0,0.001921,0.042026,0.440501,707.0,0.440501,2147.0,0.390501,1234.0,0.561345,18287.0,0.333333,38943.0
40491,disruptive_supply_chain_uncertainty,disruptive supply chain uncertainty,0.337993,0.000002,0.0,1.0,1.0,0.002335,0.045185,0.408718,880.0,0.408718,2573.0,0.358718,1497.0,0.686362,5028.0,0.250000,44198.5
37318,energy_efficiency_at_transmission,energy efficiency at transmission,0.359151,0.000002,0.0,1.0,1.0,0.002799,0.048720,0.395286,968.0,0.395286,2783.0,0.345286,1629.0,0.648527,8053.0,0.400000,34200.0
39188,low_energy_mode,low energy mode,0.569320,0.000002,0.0,1.0,1.0,0.003095,0.050983,0.392828,993.0,0.392828,2826.0,0.342828,1661.0,0.376717,47826.0,0.000000,56570.5
7884,day_ahead_electricity_market,day ahead electricity market,0.359242,0.000001,0.0,1.0,1.0,0.003058,0.050700,0.386942,1043.0,0.386942,2917.0,0.336942,1717.0,0.711026,3487.0,0.250000,44198.5
45908,continuous_transition_zone,continuous transition zone,0.375357,0.000002,1.0,0.0,1.0,0.004094,0.058598,0.381956,1078.0,0.381956,2995.0,0.331956,1772.0,0.482206,30689.0,0.333333,38943.0
37879,exponential_development_of_connected_object,exponential development of connected object,0.406984,0.000001,1.0,1.0,2.0,0.008123,0.047193,0.331647,1601.0,0.331647,4041.0,0.281647,2484.0,0.718013,3139.0,0.818182,7042.5


In [ ]:
entities_df.iloc[labels == 6].sort_values(by='avg_Score', ascending=False).head(15)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
22088,visual_walkability,visual walkability,0.126209,0.000002,0.0,1.0,1.0,0.001456,0.038480,0.855054,10.0,0.855054,295.0,0.805054,89.0,0.657206,7264.0,0.500000,26626.5
31559,crowdsourcing_visual_attention_datum,crowdsourcing visual attention datum,0.277735,0.000002,0.0,1.0,1.0,0.002035,0.042895,0.643527,179.0,0.643527,826.0,0.593527,386.0,0.712184,3427.0,0.500000,26626.5
35741,total_internal_reflection_fluorescence_configu...,total internal reflection fluorescence configu...,0.302750,0.000002,0.0,1.0,1.0,0.001778,0.040934,0.459781,609.0,0.459781,1930.0,0.409781,1099.0,0.777851,1004.0,0.600000,19752.0
17313,visual_attention_behavior,visual attention behavior,0.224161,0.000002,0.0,1.0,1.0,0.003338,0.052832,0.453696,653.0,0.453696,2004.0,0.403696,1148.0,0.689097,4841.0,0.000000,56570.5
53156,microwave_imaging_reconstruction_problem,microwave imaging reconstruction problem,0.147592,0.000002,2.0,0.0,2.0,0.013353,0.072359,0.452915,657.0,0.452915,2012.0,0.402915,1153.0,0.747899,1874.0,0.833333,6235.0
43897,three_channel_image_classification,three channel image classification,0.435902,0.000001,0.0,1.0,1.0,0.001393,0.038001,0.452800,659.0,0.905599,250.0,0.805599,88.0,0.767991,1239.0,0.777778,9188.5
25836,video_segmentation,video segmentation,0.307914,0.000003,1.0,0.0,1.0,0.001558,0.039257,0.375701,1133.0,0.375701,3093.0,0.325701,1860.0,0.518008,24862.0,0.500000,26626.5
39362,class_i_injectable_bone_substitute,class i injectable bone substitute,0.148225,0.000002,5.0,0.0,5.0,0.025289,0.069974,0.369271,1181.0,0.369271,3198.0,0.319271,1934.0,0.836271,208.0,0.000000,56570.5
41375,rapid_switching_speed,rapid switching speed,0.426197,0.000002,1.0,0.0,1.0,0.004039,0.058175,0.342793,1467.0,0.342793,3749.0,0.292793,2311.0,0.444498,37139.0,0.000000,56570.5
28664,heterogeneous_gaze_pattern,heterogeneous gaze pattern,0.434771,0.000002,0.0,1.0,1.0,0.001446,0.038405,0.331248,1609.0,0.331248,4056.0,0.281248,2494.0,0.613205,11715.0,0.000000,56570.5


In [ ]:
(labels == 52) & (entities_df['avg_Score'].lt(0.05).to_numpy())

array([False, False, False, ..., False, False, False], shape=(58198,))

In [ ]:
entities_df.iloc[(labels == 13) & (entities_df['avg_Score'].gt(0.05).to_numpy())].sort_values(by='avg_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
46995,substrate_integrated_waveguide_technology,substrate integrated waveguide technology,0.241835,5.667147e-07,1.0,0.0,1.0,0.003774,0.056156,0.742770,83.0,0.742770,518.0,0.692770,207.0,0.728609,2657.0,0.500000,26626.5
25381,designed_filter_model,designed filter model,0.440269,1.133429e-06,0.0,1.0,1.0,0.004141,0.058955,0.718876,102.0,0.718876,589.0,0.668876,240.0,0.489679,29483.0,0.500000,26626.5
23725,waveguide_filter,waveguide filter,0.273177,1.983502e-06,0.0,1.0,1.0,0.002087,0.043296,0.700705,113.0,0.700705,630.0,0.650705,261.0,0.653112,7623.0,0.500000,26626.5
45055,attractive_solution_for_filters_miniaturization,attractive solution for filters miniaturization,0.394329,1.133429e-06,0.0,1.0,1.0,0.002332,0.045158,0.697929,118.0,0.697929,637.0,0.647929,268.0,0.622621,10667.0,0.500000,26626.5
46707,advances_in_thz_communication,advances in thz communication,0.251637,5.667147e-07,1.0,0.0,1.0,0.003605,0.054871,0.680828,133.0,0.680828,692.0,0.630828,302.0,0.674102,5904.0,0.500000,26626.5
54271,high_performance_microstrip_lowpass_filter,high performance microstrip lowpass filter,0.174164,1.700144e-06,0.0,1.0,1.0,0.002244,0.044488,0.674792,137.0,0.674792,711.0,0.624792,311.0,0.783502,885.0,0.000000,56570.5
11421,different_frequency_band,different frequency band,0.488551,1.700144e-06,0.0,1.0,1.0,0.002055,0.043045,0.631248,198.0,0.631248,874.0,0.581248,421.0,0.542889,21052.0,0.500000,26626.5
51196,antimicrobial_filter_containing_zinc_pyrithione,antimicrobial filter containing zinc pyrithione,0.159614,1.700144e-06,1.0,0.0,1.0,0.001828,0.041315,0.615160,222.0,0.615160,941.0,0.565160,457.0,0.864752,63.0,0.500000,26626.5
27050,air_filled_substrate_integrated_waveguide_tech...,air filled substrate integrated waveguide tech...,0.209211,1.133429e-06,3.0,0.0,3.0,0.012650,0.049511,0.485971,516.0,0.485971,1688.0,0.435971,950.0,0.751585,1749.0,0.800000,7975.0
22735,filterbank_design,filterbank design,0.428475,1.133429e-06,0.0,1.0,1.0,0.001296,0.037260,0.474439,551.0,0.474439,1791.0,0.424439,1011.0,0.779501,970.0,0.500000,26626.5


In [79]:
from src.commands.build_emap.clustering import CommunityDetectionBasedClustering
from src.commands.build_emap.representative_nodes_selection import RepresentativeEntitiesSelector

In [ ]:
cdbc = CommunityDetectionBasedClustering(
    alg='leiden',
    graph_type='similarity',
    threshold_or_num_neighbors=0.5,
    modularity='Newman',
    resolution=1.0,
    weighted=True,
    min_cluster_size=5
)

X = results['baseline_2_final']['entities_sem_embeddings']
c = cdbc.cluster(X)

NameError: name 'CommunityDetectionBasedClustering' is not defined